
# 🚕 Taxi Fleet Performance Intelligence
## End-to-End Operational Analytics Project

**Author:** Juan Ojeda  
**Target Roles:** Data Analyst | BI Analyst | Operations Analyst  

---

## 🎯 Business Context

A metropolitan taxi fleet aims to:

- Optimize revenue distribution
- Detect operational inefficiencies
- Reduce driver idle time
- Segment drivers based on performance
- Identify peak demand windows

This project converts raw trip-level operational data into actionable business intelligence using Python and Machine Learning.



## 🛠 Technology Stack
- Python  
- Pandas  
- Matplotlib  
- Scikit-Learn  
- K-Means Clustering  


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

plt.style.use("default")


## 📥 Data Loading & Preparation

In [ ]:

df = pd.read_csv("Trips_activity.csv")

df = df.drop(columns=["Unnamed: 0"], errors="ignore")

datetime_columns = ["solicitud_viaje", "final_viaje", "inicio_turno"]
for col in datetime_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df.head()



## 📊 KPI 1: Completed Trips per Day

**Objective:** Identify high-demand days to support forecasting and staffing decisions.


In [ ]:

df_completed = df[df["viaje_estado"] == "completed"].copy()
df_completed["trip_date"] = df_completed["final_viaje"].dt.date

trips_per_day = df_completed.groupby("trip_date").size()

plt.figure(figsize=(10,6))
trips_per_day.plot(kind="bar")
plt.title("Completed Trips per Day")
plt.ylabel("Number of Trips")
plt.xticks(rotation=45)
plt.show()

trips_per_day.sort_values(ascending=False)



## 💰 KPI 2: Revenue by Hour

**Objective:** Identify revenue concentration windows to optimize driver allocation and pricing strategies.


In [ ]:

df_completed["hour"] = df_completed["final_viaje"].dt.hour

df_filtered = df_completed[
    (df_completed["viaje_precio"] > 0) &
    (df_completed["viaje_precio"] < 250)
]

revenue_per_hour = df_filtered.groupby("hour")["viaje_precio"].sum()

plt.figure(figsize=(10,6))
revenue_per_hour.plot()
plt.title("Total Revenue by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Total Revenue")
plt.show()

revenue_per_hour.sort_values(ascending=False).head(3)



## ⏳ KPI 3: Driver Idle Time Analysis

**Objective:** Measure inefficiencies between consecutive trips to identify optimization opportunities.


In [ ]:

driver_sample = df_completed[df_completed["id_conductor"] == 673].copy()
driver_sample = driver_sample.sort_values("solicitud_viaje")

driver_sample["idle_time"] = (
    driver_sample["solicitud_viaje"] -
    driver_sample["final_viaje"].shift(1)
)

driver_sample = driver_sample[
    (driver_sample["idle_time"].dt.total_seconds() > 0) &
    (driver_sample["idle_time"].dt.total_seconds() < 1800)
]

average_idle_minutes = driver_sample["idle_time"].dt.total_seconds().mean() / 60
round(average_idle_minutes, 2)



## 🤖 Driver Segmentation (Machine Learning)

Drivers are segmented into three clusters using:

- Completed Trips  
- Total Revenue  
- Distance Driven  

This enables:

- Performance benchmarking  
- Incentive strategy design  
- Underperformance detection  
- Operational optimization  


In [ ]:

driver_summary = df_completed.groupby("id_conductor").agg(
    completed_trips=("viaje_precio", "count"),
    total_revenue=("viaje_precio", "sum"),
    total_km=("viaje_distancia", "sum")
).reset_index()

X = driver_summary[["completed_trips", "total_revenue", "total_km"]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
driver_summary["cluster"] = kmeans.fit_predict(X_scaled)

silhouette_score(X_scaled, driver_summary["cluster"])



## 📌 Business Value Delivered

✔ Demand pattern detection  
✔ Revenue concentration insights  
✔ Idle time visibility  
✔ Data-driven driver segmentation  
✔ Foundation for forecasting & workforce planning  

---

This project demonstrates the ability to transform raw operational data into strategic, executive-ready insights.
